In [0]:
from pyspark.sql import functions as F
import logging
import sys
from pyspark.sql.types import StructType, StructField, TimestampType, IntegerType, FloatType
import uuid
from datetime import datetime, timezone
from pyspark.sql.window import Window
from delta.tables import DeltaTable

logger = logging.getLogger("turbines")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)
logger.propagate = False

run_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc)

logger.info(f"Gold started at {started_at} with pipeline id {run_id}")

In [0]:
def log_run(layer, started_at, rows_in, rows_out, status="SUCCESS", message=None):
    spark.createDataFrame(
        [(run_id, layer, started_at, datetime.now(timezone.utc), rows_in, rows_out, status, message)],
        "run_id string, layer string, started_at timestamp, finished_at timestamp, rows_in long, rows_out long, status string, message string"
    ).write.mode("append").saveAsTable("turbines.control.pipeline_log")

In [0]:
daily = (spark.table("turbines.silver.readings")
    .groupBy("turbine_id", F.to_date("timestamp").alias("date"))
    .agg(F.min("power_output").alias("min_mw"),
         F.max("power_output").alias("max_mw"),
         F.avg("power_output").alias("avg_mw"),
         F.count("*").alias("n_readings")))
n_in = daily.count()
logger.info(f"daily stats: {n_in} turbine-days")

In [0]:
fleet = (daily.groupBy("date")
    .agg(F.avg("avg_mw").alias("fleet_mean"),
         F.stddev("avg_mw").alias("fleet_std")))

flagged = (daily.join(fleet, "date")
    .withColumn("lower_bound", F.col("fleet_mean") - 2 * F.col("fleet_std"))
    .withColumn("upper_bound", F.col("fleet_mean") + 2 * F.col("fleet_std"))
    .withColumn("is_anomaly",
        (F.col("avg_mw") < F.col("lower_bound")) |
        (F.col("avg_mw") > F.col("upper_bound"))))

n_anomalies = flagged.filter("is_anomaly").count()
logger.info(f"anomaly flags: {n_anomalies} turbine-days outside fleet mean ± 2σ")

In [0]:
flagged.write.mode("overwrite").saveAsTable("turbines.gold.daily_stats")

In [0]:
n_gold = spark.table("turbines.gold.daily_stats").count()
spark.table("turbines.gold.daily_stats").filter("is_anomaly").show()
log_run("gold", started_at, n_in, n_gold)